# 12 — merged embryo correspondence

**Feeds:** Fig 4d, Fig 4e

**Position in the chain:** Builds the 18-label embryo taxonomy (17 of 50 clusters dropped, 33 merged), the correlation matrix, and the integration heatmap that IS the published Fig 4d.

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

This notebook ran on the earlier human embryo clustering object and the `07b` and `07c` labels built from it (`parity/results/intermediates/`), so it does not run on the embryo object `07_human_embryo.ipynb` builds here. The earlier object is in the Zenodo deposit (`../../data/DOWNLOAD.md`); `scrnaseq/run_chain.py` runs this notebook, and its `07b`/`07c` dependencies, when it is present under `SCRNASEQ_INPUT_ROOT`, and skips them otherwise. Fig 4d and 4e are drawn from its saved tables in `scrnaseq/morph_embryo_correspondence/derived/` when this notebook does not run.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original searched upward for the analysis directory (the one holding `PROVENANCE_MAP.md`, `parity/` and `trunk_main_dev/`). `ANALYSIS_ROOT` stands in for that directory: it is `$SCRNASEQ_RESULTS_ROOT` (default `scrnaseq/output/`), where the chain's notebooks write in the same layout, and this notebook's outputs go under its `experiments/rq1_5_human_embryo_alignment/`. The helper modules are imported from `scrnaseq/morph_embryo_correspondence/scripts/` instead of `experiments/rq1_5_human_embryo_alignment/src/`.

2. The embryo clusters object and its SMD companion. `SCRNASEQ_INPUT_ROOT` (default `data/scrnaseq_inputs/`) is checked for `earlier_embryo_run/results/intermediates/07_human_embryo/{adata_embryo_with_clusters,adata_embryo_SMD}.h5ad`; when absent, the notebook falls back to `embryo_path`, as before this override existed (in practice unreachable, since `run_chain.py` only runs this notebook when the object is present).

No other line of code was changed.


# 12. Final Embryo-Reassigned Cluster-Cluster Correlation

Cluster-cluster correlation figure using the current `integration_gene_panel`, canonical trunk morph clusters from `trunk_main_dev/02`, and embryo groups derived from `parity/07` with `07b` floor-plate reassignment and `07c` NMP reassignment applied only within their parent embryo clusters.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src' / 'trunk_morph_ref').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'src' / 'trunk_morph_ref').exists(), f'Could not locate repository root from {Path.cwd()}'
sys.path.insert(0, str(REPO_ROOT))
from src.trunk_morph_ref.paths import scrnaseq_input_root, scrnaseq_results_root

# Stands in for the original analysis directory: the chain's outputs, in the same layout.
ANALYSIS_ROOT = scrnaseq_results_root(REPO_ROOT)
# The earlier human embryo clusters object, fetched from the Zenodo deposit (see data/DOWNLOAD.md).
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO_ROOT)

EXPERIMENT_ROOT = ANALYSIS_ROOT / 'experiments' / 'rq1_5_human_embryo_alignment'
STAGE_NAME = '12_merged_embryo_cluster_cluster_correlation'
STAGE_DIR = EXPERIMENT_ROOT / 'results' / 'intermediates' / STAGE_NAME
FIG_DIR = STAGE_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'scrnaseq' / 'morph_embryo_correspondence' / 'scripts'))

from manuscript_cluster_cluster_correlation import (
    MORPH_DISPLAY_LABELS,
    apply_morph_display_order,
    build_gene_sets,
    cluster_averages,
    cross_cluster_correlation,
    plot_correlation_heatmap,
    plot_correlation_heatmap_label_template,
    plot_correlation_heatmap_no_axes,
    summarize_top_matches,
)
from merged_embryo_cluster_cluster_correlation import (
    DROPPED_EMBRYO_LABELS_V1,
    EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
    MERGED_EMBRYO_DISPLAY_LABELS_V3,
    MERGED_EMBRYO_ORDER_V3,
    assign_merged_embryo_labels_v3,
    load_floor_plate_reassignment_07b,
    load_nmp_reassignment_07c,
    merged_cluster_averages_v3,
    merged_embryo_membership_table_v3,
    merged_group_counts_v3,
    merged_group_summary_table_v3,
    validate_merge_scheme_v3,
)
from src.trunk_morph_ref.pipeline_io import load_h5ad, load_pickle, save_json


In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
from cmcrameri.cm import batlow

from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.plotting import (
    bold_selected_heatmap_yticklabels,
    hide_heatmap_axes_and_colorbar,
    keep_only_selected_heatmap_yticklabels,
    plot_empty_heatmap_axes,
)
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    ensure_gene_id_index,
    install_scanpy_symbol_defaults,
)

install_scanpy_symbol_defaults(sc)
plt.rcParams['svg.fonttype'] = 'none'


In [ ]:
trunk_path = ANALYSIS_ROOT / 'trunk_main_dev' / 'results' / 'intermediates' / '02_trunk_main'
embryo_path = ANALYSIS_ROOT / 'parity' / 'results' / 'intermediates' / '07_human_embryo'
floor_plate_stage_path = ANALYSIS_ROOT / 'parity' / 'results' / 'intermediates' / '07b_human_embryo_floor_plate_subclustering'
floor_plate_reassign_path = floor_plate_stage_path / 'tables' / 'ivsc_cluster_labels.csv'
nmp_stage_path = ANALYSIS_ROOT / 'parity' / 'results' / 'intermediates' / '07c_human_embryo_nmp_subclustering'
nmp_reassign_path = nmp_stage_path / 'tables' / 'nmp_cluster_labels.csv'

adata_morph_with_clusters = load_h5ad(trunk_path / 'adata_morph_with_clusters.h5ad')
adata_morph_smd = load_h5ad(trunk_path / 'adata_morph_SMD.h5ad')

# This stage ran on the earlier human embryo clustering object; nothing in this chain writes it
# to embryo_path (the parity lineage is not rebuilt here). Read it, and its SMD companion, from
# the Zenodo deposit when present, otherwise fall back to embryo_path, as before this override
# existed.
_earlier_embryo_run_path = (
    SCRNASEQ_INPUT_ROOT / 'earlier_embryo_run' / 'results' / 'intermediates' / '07_human_embryo'
)
if (_earlier_embryo_run_path / 'adata_embryo_with_clusters.h5ad').exists():
    adata_embryo_with_clusters = load_h5ad(_earlier_embryo_run_path / 'adata_embryo_with_clusters.h5ad')
    adata_embryo_smd = load_h5ad(_earlier_embryo_run_path / 'adata_embryo_SMD.h5ad')
else:
    adata_embryo_with_clusters = load_h5ad(embryo_path / 'adata_embryo_with_clusters.h5ad')
    adata_embryo_smd = load_h5ad(embryo_path / 'adata_embryo_SMD.h5ad')

integration_gene_panel = load_pickle(trunk_path / 'integration_gene_panel.pkl')
fig4_heatmap_label_genes = load_pickle(trunk_path / 'key_genes.pkl')
morph_order_all = apply_morph_display_order(list(load_pickle(trunk_path / 'celltypeorder.pkl')))
embryo_order_all = list(adata_embryo_with_clusters.obs['leiden_embryo'].cat.categories)
floor_plate_reassignment_07b = load_floor_plate_reassignment_07b(floor_plate_reassign_path)
nmp_reassignment_07c = load_nmp_reassignment_07c(nmp_reassign_path)

summary = pd.DataFrame({
    'dataset': ['morph_full', 'morph_smd', 'embryo_full', 'embryo_smd', '07b_ivsc_assignments', '07c_pnt_nmp_assignments'],
    'n_cells': [
        adata_morph_with_clusters.n_obs,
        adata_morph_smd.n_obs,
        adata_embryo_with_clusters.n_obs,
        adata_embryo_smd.n_obs,
        int(floor_plate_reassignment_07b.shape[0]),
        int(nmp_reassignment_07c.shape[0]),
    ],
    'n_genes': [
        adata_morph_with_clusters.n_vars,
        adata_morph_smd.n_vars,
        adata_embryo_with_clusters.n_vars,
        adata_embryo_smd.n_vars,
        pd.NA,
        pd.NA,
    ],
})
display(summary)
display(floor_plate_reassignment_07b.value_counts().rename_axis('merged_group').reset_index(name='n_cells'))
display(nmp_reassignment_07c.value_counts().rename_axis('merged_group').reset_index(name='n_cells'))

MANUSCRIPT_FIG_DIR = ANALYSIS_ROOT / 'trunk_main_dev' / 'results' / 'manuscript_figures'
MANUSCRIPT_FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
missing_labels, unused_labels = validate_merge_scheme_v3(embryo_order_all)
assert not missing_labels, f'Merge scheme is missing embryo labels: {missing_labels}'

membership = merged_embryo_membership_table_v3().sort_values(['status', 'merged_group', 'assignment_rule', 'embryo_label']).reset_index(drop=True)
group_summary = merged_group_summary_table_v3()

adata_embryo_with_clusters.obs['merged_embryo_v3'] = assign_merged_embryo_labels_v3(
    adata_embryo_with_clusters.obs[['leiden_embryo']],
    floor_plate_reassignment_07b,
    nmp_reassignment_07c,
).values
group_counts = merged_group_counts_v3(adata_embryo_with_clusters)
group_totals = (
    group_counts.groupby('merged_embryo_v3', observed=True)['n_cells']
    .sum()
    .rename('n_cells_total')
    .reset_index()
    .rename(columns={'merged_embryo_v3': 'merged_group'})
)
dropped_table = pd.DataFrame({'embryo_label': DROPPED_EMBRYO_LABELS_V1})

membership.to_csv(STAGE_DIR / 'merged_embryo_membership_table.csv', index=False)
group_summary.to_csv(STAGE_DIR / 'merged_embryo_group_summary.csv', index=False)
group_counts.to_csv(STAGE_DIR / 'merged_embryo_group_counts_by_source_cluster.csv', index=False)
group_totals.to_csv(STAGE_DIR / 'merged_embryo_group_cell_totals.csv', index=False)
dropped_table.to_csv(STAGE_DIR / 'dropped_embryo_clusters.csv', index=False)
save_json({
    'missing_labels': missing_labels,
    'unused_labels': unused_labels,
    'n_kept_groups': len(MERGED_EMBRYO_ORDER_V3),
    'n_dropped_clusters': len(DROPPED_EMBRYO_LABELS_V1),
    'floor_plate_override_source': str(floor_plate_reassign_path),
    'nmp_override_source': str(nmp_reassign_path),
}, STAGE_DIR / 'merge_validation.json')

display(group_summary)
display(group_totals)


## Mixed Trunk Morph + Parity Embryo Heatmap


In [ ]:
EMBRYO_MERGED_TO_MORPH_LABEL = {
    'Forebrain / Midbrain': 'Forebrain / Midbrain',
    'Hindbrain': 'Hindbrain',
    'Floor Plate': 'Floor Plate',
    'Intermediate-Ventral Spinal Cord': 'Intermediate-Ventral Spinal Cord',
    'Dorsal Spinal Cord': 'Dorsal Spinal Cord',
    'Roof Plate': 'Roof Plate',
    'Neural Crest': 'Neural Crest',
    'Immature Neuron': 'Immature Neuron',
    'Posterior NT': 'Posterior Neural Tube',
    'NMP': 'Neuromesodermal Progenitors',
    'Notochord': 'Notochord',
    'Presomitic Mesoderm': 'Presomitic Mesoderm',
    'Early Somite': 'Early Somite',
    'Mature Somite': 'Mature Somite',
    'Intermediate Mesoderm': 'Intermediate Mesoderm',
    'Lateral Plate Mesoderm': 'Lateral Plate Mesoderm',
    'Endothelial': 'Endothelial',
    'Non-Neural Ectoderm': 'Non-Neural Ectoderm',
}

HEATMAP_STAGE_BASENAME = 'merged_embryo__integration_gene_panel__mixed_heatmap'
HEATMAP_STAGE_PNG = FIG_DIR / f'{HEATMAP_STAGE_BASENAME}.png'
HEATMAP_STAGE_NOAXES_PNG = FIG_DIR / f'{HEATMAP_STAGE_BASENAME}_noaxes.png'
HEATMAP_STAGE_AXES_PDF = FIG_DIR / f'{HEATMAP_STAGE_BASENAME}_axesonly.pdf'
HEATMAP_STAGE_AXES_SVG = FIG_DIR / f'{HEATMAP_STAGE_BASENAME}_axesonly.svg'
HEATMAP_STAGE_COLORBAR = FIG_DIR / f'{HEATMAP_STAGE_BASENAME}_colorbar.png'
HEATMAP_STAGE_CELL_ORDER = STAGE_DIR / 'mixed_heatmap_cell_order.csv'
HEATMAP_STAGE_GROUP_COUNTS = STAGE_DIR / 'mixed_heatmap_group_counts.csv'

HEATMAP_MANUSCRIPT_PNG = MANUSCRIPT_FIG_DIR / 'Fig4e_heatmap_merged_data.png'
HEATMAP_MANUSCRIPT_NOAXES_PNG = MANUSCRIPT_FIG_DIR / 'Fig4e_heatmap_merged_data_noaxes.png'
HEATMAP_MANUSCRIPT_AXES_PDF = MANUSCRIPT_FIG_DIR / 'Fig4e_heatmap_merged_data_axesonly.pdf'
HEATMAP_MANUSCRIPT_AXES_SVG = MANUSCRIPT_FIG_DIR / 'Fig4e_heatmap_merged_data_axesonly.svg'
HEATMAP_MANUSCRIPT_COLORBAR = MANUSCRIPT_FIG_DIR / 'Fig4e_heatmap_merged_data_colorbar.png'

adata_embryo_heatmap = adata_embryo_with_clusters.copy()
adata_embryo_heatmap.obs['merged_embryo_fig1e'] = assign_merged_embryo_labels_v3(
    adata_embryo_heatmap.obs[['leiden_embryo']],
    floor_plate_reassignment_07b,
    nmp_reassignment_07c,
).values
adata_embryo_heatmap = adata_embryo_heatmap[adata_embryo_heatmap.obs['merged_embryo_fig1e'].notna()].copy()
adata_embryo_heatmap.obs['leiden_mixed'] = (
    adata_embryo_heatmap.obs['merged_embryo_fig1e'].map(EMBRYO_MERGED_TO_MORPH_LABEL).astype('object')
)
if adata_embryo_heatmap.obs['leiden_mixed'].isna().any():
    missing = sorted(adata_embryo_heatmap.obs.loc[adata_embryo_heatmap.obs['leiden_mixed'].isna(), 'merged_embryo_fig1e'].unique())
    raise ValueError(f'Missing embryo-to-morph merged labels: {missing}')

adata_morph_heatmap = adata_morph_with_clusters.copy()
adata_morph_heatmap.obs['leiden_mixed'] = adata_morph_heatmap.obs['leiden_morph'].astype(str).values

np.random.seed(0)
n_morph = adata_morph_heatmap.n_obs
if adata_embryo_heatmap.n_obs > n_morph:
    selected_cells = np.random.choice(adata_embryo_heatmap.obs_names, size=n_morph, replace=False)
    adata_embryo_heatmap = adata_embryo_heatmap[selected_cells].copy()

adata_mixed_heatmap = ad.concat([adata_morph_heatmap, adata_embryo_heatmap], join='inner', merge='first')
ensure_gene_id_index(adata_mixed_heatmap)
assert_gene_id_index(adata_mixed_heatmap)

adata_mixed_heatmap.obs['leiden_mixed'] = pd.Categorical(
    adata_mixed_heatmap.obs['leiden_mixed'].astype(str),
    categories=morph_order_all,
    ordered=True,
)

cell_order_mixed = ordered_cells_by_cluster_clustermap(
    adata=adata_mixed_heatmap,
    cluster_key='leiden_mixed',
    var_names=integration_gene_panel,
    figsize=(5, 5),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method='average',
    metric='jensenshannon',
)
adata_mixed_heatmap_sorted = adata_mixed_heatmap[cell_order_mixed, :].copy()

source_series = adata_mixed_heatmap_sorted.obs['source'].astype(str)
adata_mixed_heatmap_sorted.obs['tissue'] = np.where(
    source_series.eq('No Bead Morph'),
    'No Bead Morph',
    np.where(source_series.eq('BMP4 Bead Morph'), 'BMP4 Bead Morph', 'Week 3-4 Human Embryo')
)

cell_order_table = adata_mixed_heatmap_sorted.obs[['leiden_mixed', 'source', 'tissue']].copy()
cell_order_table.insert(0, 'cell_id', adata_mixed_heatmap_sorted.obs_names)
cell_order_table.to_csv(HEATMAP_STAGE_CELL_ORDER, index=False)

heatmap_group_counts = (
    adata_mixed_heatmap_sorted.obs.groupby(['tissue', 'leiden_mixed'], observed=True).size().rename('n_cells').reset_index()
)
heatmap_group_counts.to_csv(HEATMAP_STAGE_GROUP_COUNTS, index=False)

print(HEATMAP_STAGE_PNG)
print(HEATMAP_MANUSCRIPT_PNG)


In [ ]:
cells_in_order = adata_mixed_heatmap_sorted.obs_names
tissue_labels_in_order = adata_mixed_heatmap_sorted.obs.loc[cells_in_order, 'tissue']

source_color_dict = {
    'BMP4 Bead Morph': (1.0, 0.0, 0.0),
    'No Bead Morph': (0.0, 0.0, 1.0),
    'Week 3-4 Human Embryo': (1.0, 1.0, 0.0),
}

source_color_list = tissue_labels_in_order.map(source_color_dict)
if source_color_list.isnull().any():
    missing = tissue_labels_in_order[source_color_list.isnull()].unique().tolist()
    raise ValueError(f'Unmapped tissue values: {missing}')

color_array = np.array(source_color_list.tolist())[np.newaxis, :, :]
fig, ax = plt.subplots(figsize=(40, 0.5))
ax.imshow(color_array, aspect='auto')
ax.set_xticks([])
ax.set_yticks([])
for output_path in [HEATMAP_STAGE_COLORBAR, HEATMAP_MANUSCRIPT_COLORBAR]:
    fig.savefig(output_path, format='png', bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)


In [ ]:
with plt.rc_context({'figure.dpi': 300}):
    fig_dict = sc.pl.heatmap(
        adata_mixed_heatmap_sorted,
        var_names=integration_gene_panel,
        groupby='leiden_mixed',
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 40),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )

    bold_selected_heatmap_yticklabels(fig_dict, fig4_heatmap_label_genes)
    fig_dict['groupby_ax'].set_xlabel('')

    for output_path in [HEATMAP_STAGE_PNG, HEATMAP_MANUSCRIPT_PNG]:
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

with plt.rc_context({'figure.dpi': 300}):
    fig_dict = sc.pl.heatmap(
        adata_mixed_heatmap_sorted,
        var_names=integration_gene_panel,
        groupby='leiden_mixed',
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 40),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )

    hide_heatmap_axes_and_colorbar(fig_dict)

    for output_path in [HEATMAP_STAGE_NOAXES_PNG, HEATMAP_MANUSCRIPT_NOAXES_PNG]:
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()


In [ ]:
with plt.rc_context({'figure.dpi': 300}):
    ax = plot_empty_heatmap_axes(
        adata_mixed_heatmap_sorted,
        var_names=integration_gene_panel,
        groupby='leiden_mixed',
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 40),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )

    keep_only_selected_heatmap_yticklabels(ax, fig4_heatmap_label_genes)
    ax['groupby_ax'].set_xlabel('')

    for output_path in [HEATMAP_STAGE_AXES_PDF, HEATMAP_STAGE_AXES_SVG, HEATMAP_MANUSCRIPT_AXES_PDF, HEATMAP_MANUSCRIPT_AXES_SVG]:
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()


In [ ]:
gene_sets = build_gene_sets(
    adata_morph_full=adata_morph_with_clusters,
    adata_embryo_full=adata_embryo_with_clusters,
    adata_morph_smd=adata_morph_smd,
    adata_embryo_smd=adata_embryo_smd,
    integration_gene_panel=integration_gene_panel,
)
selected_gene_set = 'integration_gene_panel'
selected_genes = gene_sets[selected_gene_set]

avg_morph = cluster_averages(adata_morph_with_clusters, 'leiden_morph').loc[morph_order_all]
avg_embryo_merged = merged_cluster_averages_v3(
    adata_embryo_with_clusters,
    floor_plate_reassignment_07b,
    nmp_reassignment_07c,
    merged_order=MERGED_EMBRYO_ORDER_V3,
)
corr_df = cross_cluster_correlation(avg_morph, avg_embryo_merged, selected_genes)
corr_df = corr_df.loc[morph_order_all, [g for g in MERGED_EMBRYO_ORDER_V3 if g in corr_df.columns]]

top_matches = summarize_top_matches(corr_df, expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3)
overview = pd.DataFrame([{
    'gene_set': selected_gene_set,
    'n_genes': len(selected_genes),
    'n_morph_clusters': corr_df.shape[0],
    'n_embryo_groups': corr_df.shape[1],
    'expected_top1_rate': float(top_matches['expected_contains_top1'].dropna().mean()),
}])

pd.DataFrame([{'gene_set': k, 'n_genes': len(v)} for k, v in gene_sets.items()]).to_csv(STAGE_DIR / 'gene_set_summary.csv', index=False)
corr_df.to_csv(STAGE_DIR / 'corr_merged_embryo__integration_gene_panel.csv')
top_matches.to_csv(STAGE_DIR / 'top_matches_merged_embryo__integration_gene_panel.csv', index=False)
overview.to_csv(STAGE_DIR / 'correlation_overview.csv', index=False)

display(overview)
display(top_matches)


In [ ]:
fig = plot_correlation_heatmap(
    corr_df,
    title='Trunk morph vs human embryo cluster correlations',
    expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
    morph_label_lookup=MORPH_DISPLAY_LABELS,
    embryo_label_lookup=MERGED_EMBRYO_DISPLAY_LABELS_V3,
    x_label='Trunk morph cluster',
    y_label='Human embryo cluster',
    output_path=None,
)
fig.savefig(FIG_DIR / 'merged_embryo__integration_gene_panel.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
fig.savefig(FIG_DIR / 'merged_embryo__integration_gene_panel.pdf', dpi=300, bbox_inches='tight', pad_inches=0.02)
fig.savefig(FIG_DIR / 'merged_embryo__integration_gene_panel.svg', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()
plt.close(fig)


In [ ]:
fig_no_axes = plot_correlation_heatmap_no_axes(
    corr_df,
    expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
    output_path=FIG_DIR / 'merged_embryo__integration_gene_panel__no_axes.png',
)
plt.show()
plt.close(fig_no_axes)


In [ ]:
corr_df_zero = pd.DataFrame(0.0, index=corr_df.index, columns=corr_df.columns)
fig_template = plot_correlation_heatmap_label_template(
    corr_df_zero,
    morph_label_lookup=MORPH_DISPLAY_LABELS,
    embryo_label_lookup=MERGED_EMBRYO_DISPLAY_LABELS_V3,
    x_label='Trunk morph cluster',
    y_label='Human embryo cluster',
)
fig_template.savefig(FIG_DIR / 'merged_embryo__integration_gene_panel__label_template.pdf', dpi=300, bbox_inches='tight', pad_inches=0.02)
fig_template.savefig(FIG_DIR / 'merged_embryo__integration_gene_panel__label_template.svg', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()
plt.close(fig_template)


In [ ]:
save_json({
    'stage': STAGE_NAME,
    'morph_input': str(trunk_path / 'adata_morph_with_clusters.h5ad'),
    'morph_smd_input': str(trunk_path / 'adata_morph_SMD.h5ad'),
    'embryo_input': str(embryo_path / 'adata_embryo_with_clusters.h5ad'),
    'embryo_smd_input': str(embryo_path / 'adata_embryo_SMD.h5ad'),
    'floor_plate_override_input': str(floor_plate_reassign_path),
    'nmp_override_input': str(nmp_reassign_path),
    'merged_embryo_order': MERGED_EMBRYO_ORDER_V3,
    'dropped_embryo_clusters': DROPPED_EMBRYO_LABELS_V1,
    'selected_gene_set': selected_gene_set,
    'selected_gene_count': len(selected_genes),
    'morph_order_all': morph_order_all,
    'figure_dir': str(FIG_DIR),
    'mixed_heatmap_stage_png': str(HEATMAP_STAGE_PNG),
    'mixed_heatmap_stage_noaxes_png': str(HEATMAP_STAGE_NOAXES_PNG),
    'mixed_heatmap_stage_axes_pdf': str(HEATMAP_STAGE_AXES_PDF),
    'mixed_heatmap_stage_axes_svg': str(HEATMAP_STAGE_AXES_SVG),
    'mixed_heatmap_stage_colorbar': str(HEATMAP_STAGE_COLORBAR),
    'mixed_heatmap_manuscript_png': str(HEATMAP_MANUSCRIPT_PNG),
    'mixed_heatmap_manuscript_noaxes_png': str(HEATMAP_MANUSCRIPT_NOAXES_PNG),
    'mixed_heatmap_manuscript_axes_pdf': str(HEATMAP_MANUSCRIPT_AXES_PDF),
    'mixed_heatmap_manuscript_axes_svg': str(HEATMAP_MANUSCRIPT_AXES_SVG),
    'mixed_heatmap_manuscript_colorbar': str(HEATMAP_MANUSCRIPT_COLORBAR),
}, STAGE_DIR / 'meta.json')
print(FIG_DIR / 'merged_embryo__integration_gene_panel.png')
